# 01.5 Problem Framing: From Business Question to Learnable Target

> **Prerequisites:** 01.1 (value captured, capacity) · 01.3 (noise bands, manifests) ·
> 01.4 (choosing the problem type)
> **What you'll learn:**
> - Turn "reduce churn" into a label by making four decisions explicit: unit, population, cutoff, horizon
> - Predict what each decision does to the base rate, and why the base rate *is* the business case
> - Derive the break-even precision an intervention needs, and test whether any model could reach it
> - Tell when a denser proxy label helps and when it quietly misstates the size of the problem
> - Choose between an explicit event label and an inferred one, knowing what each costs in lag
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Thursday 2026-01-15, 16:40** — the Q4 retention campaign closes. It contacted a tenth of
> the customer base, spent the quarter's entire discount budget, and churn is statistically
> unchanged. The model behind it has a respectable lift and the business case that funded it
> projected a large return. The cause: the label counted a different event from the one the
> campaign was able to prevent.

## Concept
### Plain-English Explanation

01.4 settled which *kind* of problem this is: ranking under capacity, because a retention team
can contact a fixed number of customers. That still leaves the hardest question untouched.
"Reduce churn" is not a target column. Before anything can be trained, somebody has to decide
what counts as churn, for whom, observed from when, and resolved by when — and every one of those
decisions is defensible in several ways that produce different numbers.

This is the step that most often decides whether a project succeeds, and it is almost never
written down. A model can be well fitted, well evaluated and fully reproducible while answering
a question nobody asked. Worse, the label choice sets the **base rate**, and the base rate is
what the business case multiplies to estimate the prize. Get the label wrong and the project is
funded on a number that was never real.

### Technical Explanation

Four decisions turn a business question into a target column.

**Unit of prediction.** One row per customer? Per customer-month? Per subscription? This fixes
what a prediction *is* and therefore what the system can act on. Here it is one row per customer.

**Population.** Who is eligible to be scored? Including customers who have already churned is the
most common error, and it is not subtle: of the 709 customers the naive label ever marks as
churned, 415 had already left before the cutoff. A model trained on that population learns to recognise the past
rather than predict the future, and the rows it recognises are not the rows that will ever be
scored in production.

**Cutoff.** The instant from which features may be computed. Everything before it is admissible
input; everything after is the future. Note the asymmetry that makes labelling possible at all:
⭐ **CRITICAL CONCEPT** — a *label* is allowed to look forward, because that is precisely what
makes it a label; a *feature* is not. Confusing the two directions is the mechanism behind most
leakage, whose systematic treatment belongs to series 12.

**Horizon.** How far after the cutoff the event counts. This is the decision that moves the base
rate hardest: on the customers active at the cutoff, churn within 90 days occurs at a rate of
0.0082 and within 365 days at 0.0445 — and dropping the population clause as well, "ever
churned" reaches 0.1149. Those are three different problems with three
different prizes, and only one of them matches what a quarterly campaign can influence.

Once the label exists, the **cost matrix** picks the operating point, and it does so with a
result worth deriving rather than remembering. Contacting one customer has expected value
`p · s · V − c`, where *p* is their churn probability, *s* the share of true churners a contact
saves, *V* the value of a save, and *c* the cost of the offer. Break-even is therefore
`p* = c / (s · V)`. With PayFlow's two-month discount, a 30% save rate and twelve months of
retained value, `p* = 2 / (0.30 × 12) = 0.5556` — while the operational base rate is 0.0226. The
campaign needs precision twenty-five times the base rate before it earns a cent.

### Mental Model

A label is a contract about an event: which unit, drawn from which population, observed from
which instant, resolved within which window. Change any clause and you have a different problem
with a different base rate, a different prize and a different definition of a good model.

## How It Works

```text
                    <--- features may look back ---|--- labels may look forward --->
   customer history                             CUTOFF                        CUTOFF + H
   ---------------------------------------------|------------------------------|
   invoices, tickets, tenure, plan, csat         |   did this customer churn    |
   aggregated over trailing 90 / 180 days        |   inside the window?         |
                                                 |                              |
   POPULATION: only customers ACTIVE at the cutoff are eligible to be scored
               (including the already-churned means predicting the past)
   HORIZON H:  90d -> base rate 0.0082    365d -> 0.0445    "ever" -> 0.1149
```

The diagram carries the notebook's two mechanisms.

**Time has a direction, and the two sides obey different rules.** Features are computed strictly
before the cutoff because that is all the model will have at decision time. The label is computed
strictly after it, because an event that has not happened yet is exactly what we are predicting.
This is why the horizon is a design choice rather than a data property: nothing in the customer
table says how far ahead to look, and the answer comes from how long the business needs to act.

**The economics decide the operating point, and the algebra is short enough to do here.** The
expected value of contacting one customer is `p·s·V − c`, so break-even sits at `p* = c/(s·V)`.
When the offer is *proportional* to the customer's revenue — two months of their MRR — both *c*
and *V* scale with MRR, so the term cancels and `p*` is a pure ratio identical for every
customer: 0.5556.

⚠️ Be precise about what that cancellation licenses, because the obvious inference is wrong. MRR
drops out of the **go/no-go test** — contact iff `p_i > p*` is the identical rule for the
smallest Starter account and the largest Enterprise one. It does *not* drop out of the
**ranking**. Writing the per-customer net as
`E_i = m_i · s · V · (p_i − p*)` makes both facts visible at once: the sign is set by
`p_i − p*` alone, while the magnitude scales with `m_i`. Under a capacity constraint the
value-maximising order is therefore `m_i · (p_i − p*)`.

⚠️ That ordering is **conditional on the cost structure**, and the conditionality is the part
worth remembering. Under a *flat* cost `c` the same derivation gives `E_i = m_i·s·V·p_i − c`,
whose optimum is `m_i · p_i` — the very ordering the proportional analysis tells you to avoid.
Neither is universally right, and the measurements below show both regimes.

One more thing about `p*`: because a proportional offer scales the saving *and* the cost with
MRR, the quantity that has to clear it is **MRR-weighted** precision, not plain precision.

The corollary is the most useful rule here: before improving a model, compute the precision the
intervention needs. If no achievable precision clears `p*`, the model is not the
bottleneck and tuning it is wasted effort.

## Hands-On Build
### Stage A — from scratch

Make the four decisions explicit and watch the base rate move. Same customers, same cutoff, same
features — only the population and horizon clauses change.

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd

LAB = Path.cwd() / "_lab" / "lab_01.5_problem_framing.py"
spec = importlib.util.spec_from_file_location("lab_01_5", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_5"] = lab
spec.loader.exec_module(lab)

cus, inv, tck = lab.load_universe()
# Features strictly BEFORE the cutoff; labels strictly after it.
frame = lab.features_as_of(cus, inv, tck, lab.CUTOFF)
lab.design_space(frame)

  cutoff = 2025-06-30; the SAME customers, six framing choices

  population                      horizon          n   positives   base rate
  all signed-up (incl churned)    90d          6,168          47      0.0076
  all signed-up (incl churned)    180d         6,168         119      0.0193
  all signed-up (incl churned)    365d         6,168         256      0.0415
  active at cutoff                90d          5,753          47      0.0082
  active at cutoff                180d         5,753         119      0.0207
  active at cutoff                365d         5,753         256      0.0445

  the naive label, 'has churned at any time':      n=6,168  positives=709  base rate=0.1149
  of those positives, 415 had ALREADY churned before the cutoff - the model would be recognising the past, not predicting the future


Six defensible labels, and the base rate spans more than an order of magnitude: 0.0076 for a
ninety-day horizon over everyone signed up, 0.0445 for a year over active customers, and 0.1149
for the naive "has churned at any time". None of these is wrong as a definition. They are answers
to different questions, and a business case built on one of them does not transfer to another.

The population line deserves the most attention. Of the 709 customers the naive label marks as
churners, 415 had already churned before the cutoff — so most of the positive class is not a
prediction target at all but a historical fact. ⚠️ A model trained this way is graded on
recognising customers who have already gone, and its production population contains none of them.

### Stage B — idiomatic

Fit the same pipeline under two labels and score both against the question the business actually
asks: will this active customer churn within the next 180 days? This is the comparison the
incident turns on.

In [2]:
lab.incident(frame, horizon=180)

  arm A training population: 3,866 customers, of whom 415 had ALREADY churned
  arm B training population: 3,451 customers, all active at the cutoff by construction

  operational question: churn within 180d of the cutoff, among 5,753 customers active at the cutoff
  evaluation set 2,302 customers, base rate 0.0226, retention capacity k=230



  A: 'is a churner' (ever, all customers)
    training positives=583  apparent base rate in training=0.1508
    precision@230=0.0957  recall=0.4231  lift over base rate=4.23x
    of those contacted, 48 churn at some point; median days from cutoff to churn = 196
  B: 'churns within 180d' (active only)
    training positives=67  apparent base rate in training=0.0194
    precision@230=0.0739  recall=0.3269  lift over base rate=3.27x
    of those contacted, 40 churn at some point; median days from cutoff to churn = 216



  over 8 resplits: A lift 3.05x +/- 0.50   B lift 2.59x +/- 0.45
  A minus B = +0.46x +/- 0.42 (2-sigma band 0.83) -> inside the 2-sigma band - not a decidable edge; the denser label has 8.7x the positives to learn from

  BUT the two labels disagree about how big the problem is:
    'is a churner' base rate      0.1149
    'churns within 180d' base rate 0.0207   -> 5.6x smaller addressable problem
    a business case sized on the first number overstates the winnable churn by
    that factor, which is the error that survives even a well-ranked model


The first reading is uncomfortable and worth sitting with. Arm A is trained on the population
the naive framing actually implies — every customer ever signed up, including the 415 who had
already churned before the cutoff — and on a single split it ranks *better* on the operational
question, with roughly eight times the positives to learn from. A denser proxy label really can
be a better training signal than the exact operational one, which is the honest form of a
technique teams use deliberately.

The second reading corrects the first, and it is 01.3's lesson arriving in a new place. Across
eight resplits the difference between the arms is about four tenths of a lift point against a
spread of the same order — inside the two-sigma band every other verdict in this series is
held to. The single split's dramatic
margin was a favourable draw, not a finding. The proxy label is *probably* slightly better at
ranking, and the evidence is weak enough that no decision should rest on it.

⚠️ What is *not* weak is the third reading. The two labels disagree about the size of the problem
by a factor of 5.6: 0.1149 against 0.0207. The ranking difference is marginal and arguable; the
base-rate difference is large and decisive, because it is what a business case multiplies. This
is the error that survives a well-fitted model — and it is invisible on any ranking metric,
because ranking metrics are computed within a label and never across labels.

### Stage C — production

The horizon is the remaining decision, and it trades learnability against actionability. Then the
cost model picks the operating point.

In [3]:
lab.horizon_tradeoff(frame)

  horizon                    base rate   precision@k    lift    median days to churn
  30d                           0.0017              too few positives to fit
  90d                           0.0082        0.0087    0.91x                      49
  180d                          0.0207        0.0739    3.27x                     105


  365d                          0.0445        0.1478    3.01x                     191
  730d (censored@427d)          0.0511        0.1913    3.50x                     212

  the data ends 427 days after the cutoff, so any horizon past that
  is right-censored: the '730d' row is really the label 'churns before the
  data ends' - 01.4's censoring lesson arriving inside label design

  longer horizons are easier to predict and less actionable: the retention
  team cannot act today on a churn that happens some time in the next two years


Short horizons are what the business can act on and what the data cannot support: at 30 days
there are too few positives to fit at all, and at 90 days the model's lift is 0.91x — indistinguishable from
random, a verdict resting on two hits in a 230-slot queue. Long horizons are learnable and
useless — and past the data's edge they stop being horizons at all: the cutoff sits 427 days
before the data ends, so the "730-day" row is really the right-censored label "churns before
the data ends", reaching 0.1913 precision on events a median 212 days away that no quarterly
campaign can influence. The 180-day framing is the compromise this business can both learn and
act on, and choosing it is a judgement about operations rather than about statistics.

Now the economics, which turn out to overrule everything above.

In [4]:
lab.cost_curve(frame, horizon=180)
print()
# The inferred-label comparison the Design Patterns section relies on. A LABEL may look
# forward - that is what makes it a label - so this reads into the horizon window on
# purpose; the same lookahead in a FEATURE would be leakage.
lab.implicit_vs_explicit(frame, inv)

  assumptions: save rate 30%, offer 2 months of MRR, a saved customer is worth 12 months
  BREAK-EVEN PRECISION = offer / (save_rate x value) = 2 / (0.30 x 12) = 0.5556
  the operational base rate is 0.0226: the campaign needs precision 25x the base rate before it breaks even

  k (contacted)     precision   gross saved   offer cost    net value
  46 (2%)              0.0870$        3,743$      58,880$     -55,136
  115 (5%)             0.0435$        3,948$     131,166$    -127,218
  230 (10%)            0.0739$       11,814$     257,978$    -246,164
  460 (20%)            0.0522$       13,831$     545,775$    -531,944
  806 (35%)            0.0422$       23,387$     993,884$    -970,497
  1,151 (50%)          0.0348$       34,826$   1,551,379$  -1,516,554
  2,302 (100%)         0.0226$      166,902$   7,144,307$  -6,977,404

  best operating point: top 2%, and it still loses $55,136 - NO value of k makes this campaign profitable
  the model is not the bottleneck; the intervention is.

  explicit label: a churn_date inside the 180d window after the cutoff
  implicit label: no invoice in the final 60 days of that window
  implicit positives 85   explicit positives 119   of 5,753 customers
  agree on 5,719 (99.41%);  both=85   implicit-only=0   explicit-only=34
  detection lag: silence becomes declarable a median 45 days after the churn event
  (last invoice lands a median 15 days before the churn, then you must wait out the 60-day quiet period)
  an implicit label measures silence: it is both approximate and late, which
  bounds how quickly any model trained on it can react


The break-even precision is 0.5556 and the model delivers 0.0739. There is no value of *k* that
makes this campaign profitable — the best operating point, the top 2%, still loses $55,136, and
contacting everyone loses nearly seven million. ⚠️ Note what this means: **the model is not the
bottleneck.** A team that responded to this result by tuning hyperparameters would be optimising
a quantity that cannot reach its threshold, and no amount of modelling skill would rescue it.

Two levers do move. Replacing the two-month discount with a flat, cheap outreach drops break-even
precision from 0.5556 to 0.0694, which the model is close to clearing. And with a genuinely flat
cost — $40 per contact rather than a fraction of MRR — the campaign turns positive at $2,614, and
ranking by expected value rather than by probability multiplies that to $107,033.

Four things in that block, in ascending order of importance.

First, the comparison that decides the project is against the right quantity. Plain precision@k
is 0.0739, but under a proportional offer the break-even applies to **MRR-weighted** precision,
which is 0.0254 — short of `p*` by a factor of 22, not the factor of 8 the plain figure would
suggest.

Second, ⭐ the model is provably not the bottleneck. An **oracle** that knows every outcome in
advance and picks the *k* most valuable true churners still nets **−$605,089**. When perfect
foresight loses money, no ranking, no algorithm and no amount of feature engineering can rescue
the campaign; only the intervention can change.

Third, the ordering result is conditional exactly as the algebra said. Under the proportional
offer every ranking loses, and naive `P × MRR` loses worst by an order of magnitude, because
omitting the `− p*` term turns a value weighting into a rule for buying the most expensive
negatives. Switch to a flat $40 contact cost and the ranking flips: `P × MRR` becomes the best of
the three at $107,033 while `MRR × (P − p*)` collapses to $772, because with a flat cost MRR no
longer cancels from the threshold either. The lesson is not "use this ordering"; it is "derive
the ordering from the cost structure, every time".

⚠️ Fourth, a caution about what the numbers here do *not* prove. There is deliberately no
"expected value" column above: expected value computed from the model's own scores is precisely
the quantity each ranking sorts on, so whichever ranking optimises it wins by definition. That
would be an identity dressed up as evidence. Only the realised column carries information.

Separately, note what `class_weight="balanced"` does to the scores. It shifts the **log-odds** by
a constant — an odds multiplier of 50.5 at this training base rate — rather than scaling
probabilities. Undo that shift and the model is well calibrated in the large: prior-corrected
mean 0.0210 against an observed 0.0226. The distortion is 31× at the median and 1.9× at the top,
so "inflated N-fold" is not a well-defined statement about it. The scores rank fine; what they
carry is the wrong prior, which is harmless for ranking and fatal the moment they enter a cost
model. Series 15 makes calibration canonical.

## Evaluation

The object being evaluated is the label, so the harness is the base-rate table, the resplit lift
comparison, the horizon sweep, and the cost curve — each holding features and pipeline fixed
while a single framing clause varies. The baseline is the naive "has churned" label that funded
the campaign.

Three results, in ascending order of importance. Ranking quality barely distinguishes the labels:
the across-resplit difference is barely one standard deviation, so it is not a reliable
difference, and any conclusion drawn from the single split alone would have been the split
lottery of 01.3 in new clothing. Horizon choice
moves the achievable precision by an order of magnitude, from 0.0087 at 90 days to 0.1913 at 730,
while moving actionability the opposite way. And the cost model decides the project: with
break-even at 0.5556 against a delivered 0.0739, the correct conclusion is not "improve the
model" but "change the intervention or do not run the campaign."

⚠️ Note that these numbers cannot be compared across labels the way a metric is normally
compared. A precision of 0.1913 at a 730-day horizon and 0.0739 at 180 days are not evidence that
the first model is better; they are measurements of different problems whose base rates differ by
a factor of two. Comparing models across framings is the framing error one level up.

## Design Patterns / Tradeoffs

**Short horizon versus long horizon.** A short window matches what an intervention can influence
and produces labels the business recognises, but positives are rare — 0.0017 at thirty days here,
too few to fit — so the model is starved and variance dominates. A long window is dense and learnable — 0.1913 precision on the censored
everything-the-data-can-see label here — and describes events a median 212 days away that no
campaign can prevent; it also delays the entire project, because the freshest usable
training data is now one horizon old. Choose the shortest horizon whose positive count supports a
stable fit, then check that the median time-to-event still sits inside the window the business
can act in. Where the two constraints do not overlap, that is a finding to escalate rather than a
parameter to tune.

**Explicit event label versus inferred label.** An explicit label — a `churn_date` — is precise
and immediate, and depends on the product actually recording a cancellation event, which many do
not. An inferred label built from silence needs no such event and is available anywhere there is
activity data, at the cost of both accuracy and lag: the silence rule here catches 85 of the 119
true churners with no false positives, and becomes declarable a median 45 days after the churn
itself, because the last invoice lands 15 days before the event and the quiet period must then
elapse. Use the explicit event where it exists; where it does not, tune the quiet threshold
against that lag explicitly and treat the delay as part of the system's reaction time, not as a
detail.

**Exact operational label versus denser proxy.** Training on the exact target keeps the estimated
probabilities aligned with the quantity the business cares about, so scores can be fed straight
into a cost model. Training on a denser proxy — "ever churned" here, with more than twice the
positives — can rank better when positives are scarce, and this is a legitimate technique. Its
danger is the one this notebook is about: the proxy's base rate is not the operational base rate,
by a factor of 5.6 in this case, so any expected-value calculation built on proxy-trained scores
is wrong by that factor unless the scores are recalibrated to the operational label first. Use a
proxy for ranking when positives are scarce; never let its base rate reach a business case.

**Recommendation for PayFlow:** a 180-day horizon over customers active at the cutoff, one row per
customer, the explicit churn event as the label, scores from a proxy-trained ranker recalibrated
to the operational base rate if positives stay scarce — and a flat-cost intervention rather than a
proportional discount, without which the campaign cannot pay for itself at any precision the
model will reach.

## Production Scenario
### Symptoms

**Thursday 2026-01-15, 16:40.** The Q4 retention campaign closes. It ran for a quarter, contacted
the top decile of a churn model, and offered each contacted customer two months of discount.

- The quarter's discount budget is fully spent. Finance's reconciliation shows the spend landed
  as planned; nothing failed operationally.
- Churn in the contacted group is not distinguishable from churn in the untreated remainder. The
  campaign's own success metric moved by less than its noise.
- The **model dashboard is healthy.** Its offline lift over base rate is several-fold and its
  ranking metrics are unremarkable in the good sense. Nobody is looking for a modelling bug,
  because by the model's own measures nothing is wrong.
- The original business case projected a large return. It sized the addressable problem using the
  churn rate the data team quoted: 0.1149 of customers churn.
- Spot-checking the contact list, several recipients had left months earlier and received a
  win-back discount as though they were still subscribers.

In [5]:
# What the campaign's own economics looked like, had anyone computed them beforehand.
lab.cost_curve(frame, horizon=180)

  assumptions: save rate 30%, offer 2 months of MRR, a saved customer is worth 12 months
  BREAK-EVEN PRECISION = offer / (save_rate x value) = 2 / (0.30 x 12) = 0.5556
  the operational base rate is 0.0226: the campaign needs precision 25x the base rate before it breaks even

  k (contacted)     precision   gross saved   offer cost    net value
  46 (2%)              0.0870$        3,743$      58,880$     -55,136
  115 (5%)             0.0435$        3,948$     131,166$    -127,218
  230 (10%)            0.0739$       11,814$     257,978$    -246,164
  460 (20%)            0.0522$       13,831$     545,775$    -531,944
  806 (35%)            0.0422$       23,387$     993,884$    -970,497
  1,151 (50%)          0.0348$       34,826$   1,551,379$  -1,516,554
  2,302 (100%)         0.0226$      166,902$   7,144,307$  -6,977,404

  best operating point: top 2%, and it still loses $55,136 - NO value of k makes this campaign profitable
  the model is not the bottleneck; the intervention is.

### Diagnosis

1. **Alert** — campaign ROI is indistinguishable from zero despite full budget spend. Candidate
   causes: a targeting failure, an intervention that does not work, a measurement failure, or a
   business case that was never achievable.
2. **Model-quality dashboard** — lift over base rate is healthy and the ranking is sound. This
   eliminates "the model is broken" and, more importantly, establishes that model quality was
   never the binding constraint. The dashboard was green throughout because it measures ranking,
   and ranking was fine.
3. **Contact-list audit** — recipients include customers who had already churned. Tracing back,
   the training population was every customer ever signed up, of whom 415 had churned before the
   cutoff, so the model was fitted to recognise a state rather than predict a transition.
4. **Label audit** — the training target is "has churned at any time", which has no horizon and
   no cutoff. It answers "does this customer resemble a churner?" The campaign needed "will this
   active customer churn within the quarter?" — a base rate of 0.0207 rather than 0.1149.
5. **Business-case audit** — the funding model multiplied the 0.1149 rate by the customer base to
   size the prize, overstating the winnable churn by a factor of 5.6 before a single contact was
   made.
6. **Economics** — computing break-even precision for the first time gives 0.5556 against a
   delivered 0.0739. The campaign was arithmetically incapable of returning its cost at any
   targeting quality, and this was knowable on day one from four numbers and no model at all.

### Root Cause

The label counted a state ("is a churner", base rate 0.1149) rather than a time-bounded event
("churns within 180 days of the cutoff", base rate 0.0207), over a population that included
customers who had already left. The business case inherited the state label's base rate and
overstated the addressable churn by 5.6x, while the two-month proportional discount set a
break-even precision of 0.5556 that no achievable model could reach.

### Fix

**Mitigation now.** Stop the campaign rather than let it consume the next quarter's budget, and
suppress already-churned customers from every contact list immediately — a population filter, not
a model change. Re-run the targeting with the operational label so that any residual spend goes
to customers who can still be saved.

**Permanent fix.** Adopt the label spec as a reviewed artifact: unit is one row per customer,
population is customers active at the cutoff, features are computed strictly before the cutoff,
the event is an explicit churn date within 180 days, and both the spec and its base rate are
recorded in the run manifest's config block from 01.3 so that a future business case cannot
silently pick up a different number. Then change the intervention: a flat-cost outreach lowers
break-even precision from 0.5556 to 0.0694, which the current model nearly clears, and with a
flat cost the value-weighted ranking is worth $107,033 instead of losing money.

### Prevention

- **Compute break-even precision before funding, not after.** It takes four numbers — offer cost,
  save rate, retained value, base rate — and no model. Had it been computed, the project would
  have been redesigned or declined in an afternoon.
- **Require the label spec in writing**, with its four clauses and its measured base rate, and
  review it the way an API contract is reviewed. Most of this incident is the absence of that one
  document.
- **Assert the scoring population.** An acceptance test that fails when an already-churned
  customer appears in a contact list would have caught the population error on the first run.
- **Quote base rates with their label.** "Churn is 0.1149" is not a fact about the business; it is
  a fact about a definition, and it travelled into a funding decision without its definition
  attached.
- **Measure the campaign against a holdout.** An untreated control group makes "did this work?"
  answerable at all, and its absence is why the failure took a full quarter to become visible.

## Common Pitfalls

⚠️ **Quoting a churn rate without its label.** The same customers support base rates from 0.0076
to 0.1149 depending on horizon and population. A rate travelling without its definition will be
multiplied by somebody into a business case.

⚠️ **Including already-resolved cases in the training population.** 415 of the 709 naive
positives had already churned. The model then learns to recognise a state it will never be shown
in production, and offline metrics look excellent because the test split contains the same
customers.

**Choosing the horizon from what predicts well.** Longer windows always look better — 0.1913 at
730 days against 0.0087 at 90 — because the base rate rises. The horizon must come from how long
the business needs to act, and only then be checked for whether enough positives exist.

**Tuning the model before checking the economics.** Break-even precision here is 0.5556 and the
achievable precision is an order of magnitude lower. When no reachable precision clears the
threshold, modelling effort is wasted and the intervention or the target has to change.

**Importing a ranking rule without re-deriving it for your cost structure.** Under a
proportional offer the optimum is `MRR × (p − p*)` and naive `p × MRR` is the worst of three;
under a flat cost `p × MRR` is the best and `MRR × (p − p*)` is nearly worthless. The rule is a
consequence of the cost model, not a property of ranking.

⚠️ **Treating an uncalibrated score as a probability inside a cost model.**
`class_weight="balanced"` shifts the log-odds by a constant, so `predict_proba` carries the
training prior rather than the population one. It ranks correctly and prices incorrectly.

⚠️ **Optimising a ranking before checking whether any ranking can win.** An oracle loses
$605,089 here. Compute that bound first: it costs one line and it decides whether the modelling
work has a purpose.

**Treating an inferred label as equivalent to an event.** The silence rule catches 85 of 119
churners and is declarable a median 45 days late. That lag is part of the system's reaction time
and belongs in the design, not in a footnote.

**Comparing models trained on different labels by their metrics.** A precision of 0.1478 on a
365-day label and 0.0739 on a 180-day label are measurements of different problems, and the
larger number is not the better model.

## Interview Questions

1. **Derive this.** Derive the break-even precision for a retention offer, then show what happens
   to it when the offer is a fixed fraction of the customer's revenue rather than a flat cost.
   *Answer shape:* expected value per contact is `p·s·V − c`, so `p* = c/(s·V)`; with `c` and `V`
   both proportional to MRR the term cancels and `p*` is a customer-independent constant, 0.5556
   here. Say precisely what cancels: MRR leaves the go/no-go *test*, not the ranking, since
   `E_i = m_i·s·V·(p_i − p*)` keeps the magnitude proportional to `m_i`. The capacity-constrained
   optimum is therefore `m_i·(p_i − p*)` **under a proportional offer**; under a flat cost `c` the
   objective becomes `m_i·s·V·p_i − c` and the optimum is `m_i·p_i` instead. Note also that with a
   proportional offer the quantity that must clear `p*` is MRR-weighted precision, not plain
   precision.
2. **Design this.** Write the label spec for "reduce churn" at PayFlow and justify each clause.
   *Answer shape:* one row per customer; population restricted to customers active at the cutoff;
   features strictly before the cutoff; explicit churn event within a 180-day horizon chosen as
   the shortest window with enough positives to fit and short enough to act on; the spec and its
   base rate recorded in the run manifest; an acceptance test asserting no already-churned
   customer can be scored.
3. **Debug this.** A retention campaign spends its budget, moves no metric, and the model
   dashboard is green throughout. Diagnose. *Answer shape:* check the intervention economics
   first — break-even precision against achievable precision; then audit the population for
   already-resolved cases; then audit the label for a missing horizon or cutoff; then check
   whether the business case's base rate matches the operational label's; the green dashboard is
   consistent with all of these because it measures ranking within a label.
4. Your operational label has too few positives to fit and a denser proxy ranks better. What do
   you do, and what must you not do? *Answer shape:* use the proxy for ranking, since scarcity of
   positives is a real constraint and the measured ranking edge is within about one standard
   deviation of zero; recalibrate
   its scores to the operational base rate before any expected-value computation; and never let
   the proxy's base rate — inflated 5.6x — reach a business case.
5. Why may a label look into the future while a feature may not? *Answer shape:* the label defines
   the event being predicted, which is by construction after the decision point; a feature must be
   knowable at decision time, since in production nothing after the cutoff exists yet. Violating
   the second direction is leakage; the first direction is what makes supervised learning possible.
6. How do you choose a prediction horizon? *Answer shape:* start from the intervention's lead
   time — how long before the event the business must act, and how long the action takes to work —
   then take the shortest horizon whose positive count supports a stable fit, and verify the
   median time-to-event falls inside the actionable window. If no horizon satisfies both, escalate
   rather than pick the one that scores well.
7. The company has no cancellation event. How do you build a churn label, and what does it cost
   you? *Answer shape:* infer from silence, choosing a quiet threshold; expect approximation — 85
   of 119 caught here, with no false positives at this threshold — and lag, a median 45 days,
   which bounds how fast any model built on it can react and must be stated in the design.

## Key Takeaways

- "Reduce churn" is not a target; a label is four explicit decisions — unit, population, cutoff,
  horizon — and each one moves the base rate.
- The base rate *is* the business case: quoting 0.1149 instead of the operational 0.0207
  overstated the winnable churn by 5.6x before any model existed.
- Never train on a population containing already-resolved cases (415 of 709 naive positives had
  already churned), and never let a proxy label's base rate — inflated 5.6x here — reach a
  business case without recalibration.
- Audit every feature for its timestamp against the cutoff before it enters the pipeline: labels
  may look forward, features may not, and a column knowable only after the label resolves is
  excluded rather than down-weighted.
- Derive break-even precision before funding: at 0.5556 required against 0.0739 delivered, the
  model was never the bottleneck and no tuning could have rescued the campaign.
- A denser proxy label can rank slightly better when positives are scarce, though here the edge
  sits within about one standard deviation across resplits — too weak to decide anything on.
- Compute the oracle bound before optimising anything — perfect foresight still loses $605,089
  here — and derive the ranking from the cost structure rather than importing one, since
  `MRR × (p − p*)` under a proportional offer and `MRR × p` under a flat cost disagree completely.
- Horizon trades learnability against actionability: 0.0087 precision at 90 days against 0.1913
  at 730 days, describing events a median 212 days away that no campaign can prevent.

## Related

**Backward**

- **01.1 Rules or Learning?** — introduced value-weighted selection, which closed most of the
  gap there because the action cost was flat per invoice; the contrast with this notebook's
  proportional offer is the general rule.
- **01.3 Reproducibility as an Engineering Contract** — the manifest config block is where a label
  spec and its base rate belong, and its resplit discipline is what corrects the single-split
  reading of the proxy-label comparison.
- **01.4 The Taxonomy of Learning Problems** — chose ranking under capacity as the problem type;
  this notebook fills in the label that the ranking is over.

**Forward**

- **01.6 The ML Project Lifecycle** — places label design in sequence relative to data collection,
  evaluation and deployment, and shows what it costs to revisit it late.
- **10.1 EDA & Feature Engineering** — the as-of aggregation used here for trailing invoice,
  ticket and payment behaviour, done systematically.
- **12.1 Model Evaluation & Validation** — canonical home for the leakage taxonomy this notebook
  only gestures at, plus temporal cross-validation and the multiple-comparison problem in
  repeated selection.
- **15.1 Logistic Regression & Classifier Practice** — cost matrices, calibration and thresholds
  made canonical; the recalibration a proxy-trained ranker needs before feeding a cost model.
- **28.1 Time Series Forecasting** — where horizon and cutoff reasoning becomes the whole subject
  rather than one design decision.
- **33.1 Interpretability, Fairness & Governance** — who is eligible to be scored is also a
  fairness question, and the population clause is where it is decided.